# Solution E1 · pd.get_dummies on training and test: same size, different columns

**Type**: Find the bug

## Objective

Find out why a model with an excellent AUC during development produces predictions worse than chance on new data, even though there is
no error or warning: the one-hot columns of the test set do not match those of the training set. Fix it in two ways.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder


def generate(n, regions, seed):
    r = np.random.default_rng(seed)
    reg = r.choice(regions, n)
    eff = {"Center": 2.0, "North": -2.0, "South": 0.0, "Sardinia": 1.0}
    x = r.normal(size=n)
    logit = np.array([eff[k] for k in reg]) + 1.0 * x
    return pd.DataFrame({"region": reg, "x": x}), r.binomial(1, 1 / (1 + np.exp(-logit)))


X_tr, y_tr = generate(800, ["Center", "North", "South"], 1)
X_te, y_te = generate(300, ["North", "Sardinia", "South"], 2)      # in production: a new region, and a known one that does not show up

## Commented solution

In [ ]:
# the colleague's code: it runs, but it is wrong
Xd_tr = pd.get_dummies(X_tr)
Xd_te = pd.get_dummies(X_te)
print(Xd_tr.shape, Xd_te.shape)

model = LogisticRegression(max_iter=1000).fit(Xd_tr.to_numpy(dtype=float), y_tr)
auc_bug = roc_auc_score(y_te, model.predict_proba(Xd_te.to_numpy(dtype=float))[:, 1])
print(f"test AUC: {auc_bug:.3f}")

**Where the bug is.** `pd.get_dummies` creates a column for every category **present in the data it receives**. The training set has
`region_Center`, `region_North`, `region_South`; the test set has `region_North`, `region_Sardinia`, `region_South`. The matrices both
have 4 columns (with `x`), but the second column of the training set is Center and that of the test set is North, the third is North versus
Sardinia: the model applies the coefficient of Center (+2) to the rows of North (which are worth −2), and that of North to Sardinia.

In [ ]:
different_columns = list(Xd_tr.columns) != list(Xd_te.columns)
print("training:", list(Xd_tr.columns))
print("test    :", list(Xd_te.columns))

In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse=False).fit(X_tr[["region"]])          # columns decided by the TRAINING set


def matrix(X):
    return np.column_stack([X[["x"]].to_numpy(), ohe.transform(X[["region"]])])


model_ok = LogisticRegression(max_iter=1000).fit(matrix(X_tr), y_tr)
auc_ok = roc_auc_score(y_te, model_ok.predict_proba(matrix(X_te))[:, 1])

Xd_te_aligned = Xd_te.reindex(columns=Xd_tr.columns, fill_value=0)
auc_reindex = roc_auc_score(y_te, LogisticRegression(max_iter=1000).fit(Xd_tr.to_numpy(dtype=float), y_tr)
                            .predict_proba(Xd_te_aligned.to_numpy(dtype=float))[:, 1])
print(f"AUC: with the bug {auc_bug:.3f} | OneHotEncoder {auc_ok:.3f} | reindex {auc_reindex:.3f}")
answer = "a"

## Check

In [ ]:
with verify():
    check(different_columns is True and list(Xd_tr.columns) != list(Xd_te.columns), "the columns of the test set are not those of the training set",
          "compare list(Xd_tr.columns) with list(Xd_te.columns): different_columns must be True")
    check(auc_ok > auc_bug + 0.05, "the OneHotEncoder fitted on the training set clearly improves the test AUC",
          "auc_ok must come from an encoder fitted ONLY on X_tr and applied with transform to X_te")
    check(abs(auc_ok - auc_reindex) < 1e-9, "the reindex on the training columns is equivalent to the OneHotEncoder",
          "get_dummies(X_te).reindex(columns=Xd_tr.columns, fill_value=0) must give the same result")
    check(answer == "a", "correct explanation", "reread: what changes between the two matrices, besides the number of columns?")
done()

## What you learned

- The columns of an encoder must be **decided by the training set and rebuilt identically** at every other moment: with an
  encoder with state (`OneHotEncoder`) it is automatic, with `pd.get_dummies` it is not.
- The bug produces no errors: the shapes coincide by chance (here 4 columns in both matrices). In scikit-learn 1.2 or later, with
  DataFrame input, the check on the feature names would reveal it with an error; in 1.1 it stays silent. Don't count on it: it is a case where
  a test on the columns (`assert list(...) == list(...)`) saves hours.
- With `handle_unknown="ignore"` the new region (`Sardinia`) produces zeros: the model does not know it, it is the prudent choice. The cost
  is visible in the correct result too: the AUC stays lower than what is achievable with all the information, because there is no
  information about Sardinia.

## Common pitfall

Calling `pd.get_dummies` "for convenience" during development and forgetting that in production the rows arrive one at a time
(one region at a time, so a single column!). The code that works on a whole test set breaks silently,
or obviously, on a single request.

## Interview question

<details><summary>Why do you use `OneHotEncoder` and not `pd.get_dummies` in a production pipeline?</summary>

Because the former is a transformer with state: the categories are learned from the training set and saved together with the model, so
the columns are always the same and new categories have a defined behavior (zeros, or an error if I want one).
`pd.get_dummies` is a function with no memory: the result depends on the data I pass it, and with a single request
it would produce a single column. Inside a Pipeline, moreover, it guarantees that training and production use the same code.

</details>